In [1]:
import pyomo.environ as pyo

# Setting Up

In [2]:
sources = ['Konabari', 'BagherBazar', 'Chittagong']
destinations = ['Sylhet', 'Bogura', 'Chittagong', 'Khulna', 'Gazipur']

In [3]:
# Supply (cartons)
supply = {
    'Konabari': 45000,
    'BagherBazar': 90000,
    'Chittagong': 80000
}

# Demand (cartons)
demand = {
    'Sylhet': 52000,
    'Bogura': 35000,
    'Chittagong': 50000,
    'Khulna': 50000,
    'Gazipur': 60000
}

In [4]:
# Transportation cost matrix
cost = {
    ('Konabari', 'Sylhet'): 35480,
    ('Konabari', 'Bogura'): 23380,
    ('Konabari', 'Chittagong'): 43280,
    ('Konabari', 'Khulna'): 38900,
    ('Konabari', 'Gazipur'): 2940,

    ('BagherBazar', 'Sylhet'): 35900,
    ('BagherBazar', 'Bogura'): 27160,
    ('BagherBazar', 'Chittagong'): 45940,
    ('BagherBazar', 'Khulna'): 41700,
    ('BagherBazar', 'Gazipur'): 5600,

    ('Chittagong', 'Sylhet'): 52920,
    ('Chittagong', 'Bogura'): 62040,
    ('Chittagong', 'Chittagong'): 2800,
    ('Chittagong', 'Khulna'): 67300,
    ('Chittagong', 'Gazipur'): 39960
}

# Modelling

In [5]:
#model initialization
model = pyo.ConcreteModel()
model.x = pyo.Var(sources, destinations, domain = pyo.NonNegativeReals)

### Objective: Minimizing Cost

In [6]:
def obj_rule(m):
    return sum(cost[i,j]*m.x[i,j] for i in sources for j in destinations)

In [7]:
model.obj = pyo.Objective(rule = obj_rule, sense = pyo.minimize)

### Constraints

* `supply contraints`

In [8]:
def supply_rule(m,i):
    return sum(m.x[i,j] for j in destinations) == supply[i]

model.supply_constraint = pyo.Constraint(sources, rule = supply_rule)

* `demand constraints`

In [9]:
def demand_rule(m,j):
    return sum(m.x[i,j] for i in sources) <= demand[j]

model.demand_constraint = pyo.Constraint(destinations, rule = demand_rule)

### Solution

In [10]:
solver = pyo.SolverFactory('gurobi')
result = solver.solve(model, tee=False)

In [11]:
print(f"Status      : {result.solver.termination_condition}")
print(f"Minimum Cost: Tk {pyo.value(model.obj):,.0f}\n")

print("Optimal Transportation Plan:\n")
for i in sources:
    for j in destinations:
        val = pyo.value(model.x[i, j])
        if val > 0:
            print(f"{i} -> {j}: {int(val)}")

Status      : optimal
Minimum Cost: Tk 4,394,300,000

Optimal Transportation Plan:

Konabari -> Bogura: 35000
Konabari -> Khulna: 10000
BagherBazar -> Sylhet: 22000
BagherBazar -> Khulna: 8000
BagherBazar -> Gazipur: 60000
Chittagong -> Sylhet: 30000
Chittagong -> Chittagong: 50000


In [12]:
#Supply & Demand Status Summary
total_supply = sum(supply.values())
total_demand = sum(demand.values())
unmet_demand = total_demand - total_supply

print(f"  {'Total Supply =':>15}  {total_supply:>10,}")
print(f"  {'Total Demand =':>15}  {total_demand:>10,}")
print(f"  {'Unmet Demand =':>15}  {unmet_demand:>10,}")

   Total Supply =     215,000
   Total Demand =     247,000
   Unmet Demand =      32,000


### Final Comments: Unbalanced Transportation Problem

Total supply (215,000 cartons) is **less than** total demand (247,000 cartons),
creating a shortage of **32,000 cartons**.

This makes it an **unbalanced transportation problem** with excess demand, which
has two direct implications for the problem formulation (with Pyomo):

- **Supply constraints use `==`** — every carton produced at every factory
  must be shipped out. No factory retains stock.
- **Demand constraints use `<=`** — since supply is insufficient to meet all
  demand, distribution centers receive *at most* their demanded quantity.
  The solver decides which DC absorbs the shortfall.

In the classical manual method (Vogel's, MODI), this would require adding a
**dummy supply node** of 32,000 cartons with zero shipping cost to artificially
balance the table before solving. Pyomo handles this natively through the
inequality constraints — no dummy node is needed.

The optimal solution reveals that **Khulna DC absorbs the entire 32,000-carton
shortfall**, receiving only 18,000 of its 50,000 demanded cartons (36%),
because all routes serving Khulna are among the most expensive in the network.